In [1]:
import json
import time
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from collections import defaultdict

In [2]:
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

In [3]:
class ExtractedEntities(BaseModel):
    Mythical_Entity: Optional[List[str]] = Field(default=None)
    Celestial_Entity: Optional[List[str]] = Field(default=None)
    Phenomenon: Optional[List[str]] = Field(default=None)
    Time: Optional[List[str]] = Field(default=None)
    Food: Optional[List[str]] = Field(default=None)
    Activity: Optional[List[str]] = Field(default=None)
    Concept: Optional[List[str]] = Field(default=None)
    Object: Optional[List[str]] = Field(default=None)
    Living_Being: Optional[List[str]] = Field(default=None)
    Text: Optional[List[str]] = Field(default=None)
    Location: Optional[List[str]] = Field(default=None)
    Primordial_Element: Optional[List[str]] = Field(default=None)
    Medical_Concept: Optional[List[str]] = Field(default=None)
    Geographical_Feature: Optional[List[str]] = Field(default=None)
    Event: Optional[List[str]] = Field(default=None)
    Emotions: Optional[List[str]] = Field(default=None)
    Body_part: Optional[List[str]] = Field(default=None)
    Sanskrit_text: Optional[List[str]] = Field(default=None)
    Social_Group_and_Role: Optional[List[str]] = Field(
        default=None,
        alias="Social_Group_&_Role"
    )
    Other: Optional[List[str]] = Field(default=None, description="Use this category only if the entity absolutely doesn't fit into any other categories.")

    model_config = {"populate_by_name": True}

class ExtractionResults(BaseModel):
    # 🔥 CRITICAL: The scratchpad MUST be the first field.
    step_by_step_reasoning: str = Field(
        description="Think step-by-step. Read the sentence and the list of entities. Identify which entities belong to which group and think why. remember a single entity can belong to multiple groups"
    )
    categories: ExtractedEntities = Field(
        description="An dictionary of all existing categories in the provided sentence mapped to a list of entities that belong to a given category",
        default_factory=list
    )

In [4]:
def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content

In [7]:
import json

FILE = "processing_state.json"
CATEGORY = "Social_Group_&_Role"

# category mapping
cat_dict = {
    0: 'Time',
    1: 'Activity',
    2: 'Location',
    3: 'Other',
    4: 'Food',
    5: 'Object',
    6: 'Text',
    7: 'Medical_Concept',
    8: 'Sanskrit_text',
    9: 'Social_Group_&_Role',
    10: 'Phenomenon',
    11: 'Concept',
    12: 'Event',
    13: 'Primordial_Element',
    14: 'Geographical_Feature',
    15: 'Emotions',
    16: 'Body_part',
    17: 'Living_Being',
    18: 'Celestial_Entity',
    19: 'Mythical_Entity'
}


# ----------------------------
# Load
# ----------------------------
with open(FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

registry = data["global_entity_registry"]
category_list = registry[CATEGORY]

print(f"Total entities in {CATEGORY}: {len(category_list)}\n")


# ----------------------------
# Interactive loop
# ----------------------------
for idx, entity in enumerate(category_list[:]):  # copy to avoid iteration issues

    print("\n----------------------------------")
    print(f"[{idx+1}/{len(category_list)}] Entity: {entity}")
    print(f"Other categories: {entt_to_cat.get(entity)}")
    print("Categories:")

    for k, v in cat_dict.items():
        print(f"{k}: {v}")

    user_input = input(
        "\nEnter category numbers (space separated), "
        "'s' = skip, 'x' = remove from current: "
    ).strip()

    # ----------------------------
    # Skip (keep as is)
    # ----------------------------
    if user_input.lower() == "s":
        print("→ kept in same category")
        continue

    # ----------------------------
    # Remove from current category
    # ----------------------------
    if user_input.lower() == "x":
        registry[CATEGORY] = [
            e for e in registry[CATEGORY] if e != entity
        ]

        with open(FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        print("→ removed from current category")
        continue

    # ----------------------------
    # Parse category input
    # ----------------------------
    try:
        selected = list(map(int, user_input.split()))
    except:
        print("Invalid input, skipping...")
        continue

    # ----------------------------
    # Remove from current category
    # ----------------------------
    registry[CATEGORY] = [
        e for e in registry[CATEGORY] if e != entity
    ]
    
    # ----------------------------
    # Assign to selected categories
    # ----------------------------
    for cat_id in selected:
        cat_name = cat_dict[cat_id]

        if cat_name not in registry:
            registry[cat_name] = []

        if entity not in registry[cat_name]:
            registry[cat_name].append(entity)


    # ----------------------------
    # Save immediately
    # ----------------------------
    with open(FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print("→ reassigned and saved")


print("\nDONE.")

Total entities in Social_Group_&_Role: 443


----------------------------------
[1/443] Entity: पाप
Other categories: ['Activity', 'Concept']
Categories:
0: Time
1: Activity
2: Location
3: Other
4: Food
5: Object
6: Text
7: Medical_Concept
8: Sanskrit_text
9: Social_Group_&_Role
10: Phenomenon
11: Concept
12: Event
13: Primordial_Element
14: Geographical_Feature
15: Emotions
16: Body_part
17: Living_Being
18: Celestial_Entity
19: Mythical_Entity
→ removed from current category

----------------------------------
[2/443] Entity: पकना
Other categories: []
Categories:
0: Time
1: Activity
2: Location
3: Other
4: Food
5: Object
6: Text
7: Medical_Concept
8: Sanskrit_text
9: Social_Group_&_Role
10: Phenomenon
11: Concept
12: Event
13: Primordial_Element
14: Geographical_Feature
15: Emotions
16: Body_part
17: Living_Being
18: Celestial_Entity
19: Mythical_Entity
→ reassigned and saved

----------------------------------
[3/443] Entity: क्षत्रिय
Other categories: []
Categories:
0: Time
1: Acti

In [16]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# TIME TREE
# ============================================================
TIME_LEAF_NODES = [
    "day",
    "special_date",
    "generic_date",
    "month",
    "moon_phase",
    "measurement_unit",
    "measurement_method",
    "nakshatra",
    "other"
]

PARENT_MAP = {
    "day": ["Time", "day"],
    "special_date": ["Time", "date", "special_date"],
    "generic_date": ["Time", "date", "generic_date"],
    "month": ["Time", "month"],
    "moon_phase": ["Time", "moon_phase"],
    "measurement_unit": ["Time", "measurement_unit"],
    "measurement_method": ["Time", "measurement_method"],
    "nakshatra": ["Time", "nakshatra"],
    "other": ["Time", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/time.txt", 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class TimeClassification(BaseModel):
    leaf_category: Literal[
        "day",
        "special_date",
        "generic_date",
        "month",
        "moon_phase",
        "measurement_unit",
        "measurement_method",
        "nakshatra",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 256,
        "temperature": 0.1,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=TimeClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in TIME_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

time_entities = file["global_entity_registry"]["Time"]

print(f"Total Time entities: {len(time_entities)}")

# ============================================================
# LOAD OUTPUT FILES
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in time_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Time"]
        }

        entities[node_id] = entity

    # ----------------------------
    # skip if already classified
    # ----------------------------
    existing_types = set(entity.get("types", []))
    if len(existing_types) > 1:
        print("→ already classified")
        continue

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    # ----------------------------
    # update types using full path
    # ----------------------------
    full_path = PARENT_MAP[leaf]

    types = set(entity.get("types", []))
    types.update(full_path)

    entity["types"] = list(types)

    # ----------------------------
    # save immediately
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

print("\nDONE.")

Total Time entities: 113

Processing: रामनवमी
→ special_date

Processing: कला
→ measurement_unit

Processing: अक्षयनवमी
→ special_date

Processing: प्रतिपदा
→ generic_date

Processing: मास
→ month

Processing: त्रुटि
→ measurement_unit

Processing: दिवस
→ day

Processing: सोमवार
→ day

Processing: चैत्र मास
→ month

Processing: दिन
→ day

Processing: ग्रीष्म ऋतु
→ other

Processing: अयन
→ measurement_method

Processing: नाड़ी
→ measurement_unit

Processing: रिक्ताथी
→ special_date

Processing: संवत्सरादि
→ measurement_unit

Processing: अधिक मास
→ month

Processing: योग
→ other

Processing: प्रहर
→ measurement_unit

Processing: शुक्लपक्ष
→ moon_phase

Processing: वर्षभर
→ measurement_unit

Processing: कृष्ण
→ moon_phase

Processing: एकादशी
→ special_date

Processing: वैशाख मास
→ month

Processing: रात्रि
→ measurement_unit

Processing: मासारम्भ
→ month

Processing: काष्ठा
→ nakshatra

Processing: रात
→ measurement_unit

Processing: नृसिंहचतुर्दशी
→ special_date

Processing: सूर्योदय
→ o

In [17]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# ACTIVITY TREE
# ============================================================
ACTIVITY_LEAF_NODES = [
    "festival",
    "ritual_activity",
    "mundane_activity",
    "other"
]

PARENT_MAP = {
    "festival": ["Activity", "festival"],
    "ritual_activity": ["Activity", "ritual_activity"],
    "mundane_activity": ["Activity", "mundane_activity"],
    "other": ["Activity", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/activity.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class ActivityClassification(BaseModel):
    leaf_category: Literal[
        "festival",
        "ritual_activity",
        "mundane_activity",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=ActivityClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in ACTIVITY_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

activity_entities = file["global_entity_registry"].get("Activity", [])

print(f"Total Activity entities: {len(activity_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in activity_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Activity"]
        }

        entities[node_id] = entity

    # ----------------------------
    # skip if already has Activity subtype
    # ----------------------------
    existing_types = set(entity.get("types", []))
    if any(t in ACTIVITY_LEAF_NODES for t in existing_types):
        print("→ already classified")
        continue

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    # ----------------------------
    # update types (full path)
    # ----------------------------
    full_path = PARENT_MAP[leaf]

    types = set(entity.get("types", []))
    types.update(full_path)

    entity["types"] = list(types)

    # ----------------------------
    # save immediately
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

print("\nDONE.")

Total Activity entities: 224

Processing: रामनवमी
→ festival

Processing: पाप
→ other

Processing: चढ़ावे
→ ritual_activity

Processing: तप
→ ritual_activity

Processing: स्कन्द-यात्रा
→ festival

Processing: नमोस्तु
→ ritual_activity

Processing: उत्पात
→ other

Processing: तिलकव्रत
→ ritual_activity

Processing: कार्य
→ mundane_activity

Processing: कर्म
→ mundane_activity

Processing: दाह
→ other

Processing: काम
→ mundane_activity

Processing: निर्णय
→ other

Processing: पञ्चामृत स्नान
→ ritual_activity

Processing: अर्घ्य
→ ritual_activity

Processing: सेतुबन्धन
→ ritual_activity

Processing: तपस्या
→ ritual_activity

Processing: प्रकट होना
→ mundane_activity

Processing: मालिश
→ mundane_activity

Processing: अभ्यास
→ mundane_activity

Processing: भोजन करना
→ mundane_activity

Processing: हत्या
→ other

Processing: झगड़ा
→ mundane_activity

Processing: आराधन
→ ritual_activity

Processing: प्राप्ति
→ mundane_activity

Processing: उपयोग
→ mundane_activity

Processing: उतरना
→ mundan

In [ ]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# PHENOMENON TREE
# ============================================================
PHENOMENON_LEAF_NODES = [
    "celestial_phenomenon",
    "season",
    "natural_phenomenon",
    "other"
]

PARENT_MAP = {
    "celestial_phenomenon": ["Phenomenon", "celestial_phenomenon"],
    "season": ["Phenomenon", "season"],
    "natural_phenomenon": ["Phenomenon", "natural_phenomenon"],
    "other": ["Phenomenon", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/phenomenon.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class PhenomenonClassification(BaseModel):
    leaf_category: Literal[
        "celestial_phenomenon",
        "season",
        "natural_phenomenon",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=PhenomenonClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in PHENOMENON_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

phenomenon_entities = file["global_entity_registry"].get("Phenomenon", [])

print(f"Total Phenomenon entities: {len(phenomenon_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in phenomenon_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Phenomenon"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify (always run)
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]

    existing_types = set(entity.get("types", []))

    # ----------------------------
    # check if new type already exists
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    # ----------------------------
    # add new types
    # ----------------------------
    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save only when modified
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")


Processing: ऋतुकाल
→ season
→ no new type to add

Processing: सूर्य ग्रहण
→ celestial_phenomenon
→ no new type to add

Processing: पुनर्जन्म
→ other
→ no new type to add

Processing: वैशाख
→ season
→ no new type to add

Processing: वृद्धिक्षय
→ other
→ no new type to add

Processing: जन्म
→ other
→ no new type to add

Processing: हेमन्त
→ season
→ no new type to add

Processing: वसन्तारम्भ
→ season
→ no new type to add

Processing: अपुनरागमन
→ other
→ no new type to add

Processing: वसंत ऋतु
→ season
→ no new type to add

Processing: सूर्योदय
→ celestial_phenomenon
→ updated

Processing: प्रादुर्भाव
→ other
→ updated

Processing: वसन्त ऋतु
→ season
→ no new type to add

Processing: ऋतु
→ season
→ updated

Processing: वर्षा
→ season
→ no new type to add

Processing: परिवर्तन
→ other
→ no new type to add

Processing: ऋतुपरिवर्तन
→ season
→ no new type to add

Processing: शिशिर
→ season
→ no new type to add

Processing: मौसम
→ season
→ no new type to add

Processing: ऋतु परिवर्तन
→ seaso

In [4]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# MEDICAL TREE
# ============================================================
MEDICAL_LEAF_NODES = [
    "disease",
    "symptom_physical",
    "symptom_mental",
    "secretion_internal",
    "secretion_external",
    "remedy",
    "other"
]

PARENT_MAP = {
    "disease": ["Medical_Concept", "disease"],
    "symptom_physical": ["Medical_Concept", "symptom_physical"],
    "symptom_mental": ["Medical_Concept", "symptom_mental"],
    "secretion_internal": ["Medical_Concept", "secretion_internal"],
    "secretion_external": ["Medical_Concept", "secretion_external"],
    "remedy": ["Medical_Concept", "remedy"],
    "other": ["Medical_Concept", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/medical_concept.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class MedicalClassification(BaseModel):
    leaf_category: Literal[
        "disease",
        "symptom_physical",
        "symptom_mental",
        "secretion_internal",
        "secretion_external",
        "remedy",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=MedicalClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in MEDICAL_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

medical_entities = file["global_entity_registry"].get("Medical_Concept", [])

print(f"Total Medical entities: {len(medical_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in medical_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Medical_Concept"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)

    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]

    existing_types = set(entity.get("types", []))

    # ----------------------------
    # only add if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Medical entities: 50

Processing: चर्म-सम्बन्धी विकारों
→ disease
→ updated

Processing: दोषाधक इन्द्रिय प्रवृत्ति
→ other
→ updated

Processing: अभ्यङ्ग
→ remedy
→ updated

Processing: गन्ध
→ other
→ updated

Processing: थकावट
→ symptom_physical
→ updated

Processing: मल
→ secretion_external
→ updated

Processing: वीर्य
→ secretion_internal
→ updated

Processing: ऊँघ
→ symptom_mental
→ updated

Processing: बुखार
→ disease
→ updated

Processing: लक्षण
→ other
→ updated

Processing: निद्रा
→ symptom_mental
→ updated

Processing: रुधिर
→ secretion_internal
→ updated

Processing: पाचन रस
→ secretion_internal
→ updated

Processing: व्याधि
→ disease
→ updated

Processing: जलन
→ symptom_physical
→ updated

Processing: पसीना
→ secretion_external
→ updated

Processing: प्यास
→ symptom_physical
→ updated

Processing: आयुर्वेद
→ other
→ updated

Processing: उदर शुद्धि
→ remedy
→ updated

Processing: श्वास
→ symptom_physical
→ updated

Processing: मनोवृत्ति
→ symptom_mental
→ updated

Proce

In [6]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# MYTHICAL TREE
# ============================================================
MYTHICAL_LEAF_NODES = [
    "metaphysical_entity",
    "deity",
    "avatar",
    "being_class",
    "individual_figure",
    "mythical_creature_object",
    "other"
]

PARENT_MAP = {
    "metaphysical_entity": ["Mythical_Entity", "metaphysical_entity"],
    "deity": ["Mythical_Entity", "deity"],
    "avatar": ["Mythical_Entity", "avatar"],
    "being_class": ["Mythical_Entity", "being_class"],
    "individual_figure": ["Mythical_Entity", "individual_figure"],
    "mythical_creature_object": ["Mythical_Entity", "mythical_creature_object"],
    "other": ["Mythical_Entity", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/mythical_entity.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class MythicalClassification(BaseModel):
    leaf_category: Literal[
        "metaphysical_entity",
        "deity",
        "avatar",
        "being_class",
        "individual_figure",
        "mythical_creature_object",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=MythicalClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in MYTHICAL_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

mythical_entities = file["global_entity_registry"].get("Mythical_Entity", [])

print(f"Total Mythical entities: {len(mythical_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in mythical_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Mythical_Entity"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # only update if new info
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Mythical entities: 116

Processing: भगवान
→ deity
→ updated

Processing: कामधेनु
→ mythical_creature_object
→ updated

Processing: कृष्णलीला
→ individual_figure
→ updated

Processing: कला
→ being_class
→ updated

Processing: अनन्तशक्तिमान् परब्रह्म
→ metaphysical_entity
→ updated

Processing: जगत्स्वामी
→ deity
→ updated

Processing: काम
→ deity
→ updated

Processing: भगवान कृष्ण
→ deity
→ updated

Processing: प्रेत
→ being_class
→ updated

Processing: परब्रह्म परमात्मा श्रीरामचन्द्र
→ metaphysical_entity
→ updated

Processing: पृथु
→ avatar
→ updated

Processing: अग्नि
→ deity
→ updated

Processing: श्रीरामचन्द्रजी
→ deity
→ updated

Processing: नाड़ी
→ other
→ updated

Processing: शिवजी
→ deity
→ updated

Processing: पूर्णावतार
→ avatar
→ updated

Processing: भगवान नृसिंह
→ avatar
→ updated

Processing: कार्तिकेय
→ deity
→ updated

Processing: अधिकारी अवतार
→ avatar
→ updated

Processing: परमेश्वर
→ deity
→ updated

Processing: ब्रह्मा
→ deity
→ updated

Processing: केशव
→ deit

In [7]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# LIVING BEING TREE
# ============================================================
LIVING_LEAF_NODES = [
    "human_generic",
    "human_individual",
    "animal",
    "plant",
    "mythical_living_being",
    "other"
]

PARENT_MAP = {
    "human_generic": ["Living_Being", "human_generic"],
    "human_individual": ["Living_Being", "human_individual"],
    "animal": ["Living_Being", "animal"],
    "plant": ["Living_Being", "plant"],
    "mythical_living_being": ["Living_Being", "mythical_living_being"],
    "other": ["Living_Being", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/living_being.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class LivingBeingClassification(BaseModel):
    leaf_category: Literal[
        "human_generic",
        "human_individual",
        "animal",
        "plant",
        "mythical_living_being",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=LivingBeingClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in LIVING_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

living_entities = file["global_entity_registry"].get("Living_Being", [])

print(f"Total Living_Being entities: {len(living_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in living_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Living_Being"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # add only if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Living_Being entities: 66

Processing: मानव
→ human_generic
→ updated

Processing: प्राणी
→ animal
→ updated

Processing: पृथु
→ mythical_living_being
→ updated

Processing: पूर्वज
→ human_generic
→ updated

Processing: मनु महाराज
→ human_individual
→ updated

Processing: महर्षि अगस्त्य
→ human_individual
→ updated

Processing: देहधारी
→ human_generic
→ updated

Processing: लता
→ plant
→ updated

Processing: लोग
→ human_generic
→ updated

Processing: पूर्णावतार
→ mythical_living_being
→ updated

Processing: अगस्त्य मुनि
→ human_individual
→ updated

Processing: गौरी
→ mythical_living_being
→ updated

Processing: कृष्ण
→ mythical_living_being
→ updated

Processing: वानर
→ animal
→ updated

Processing: साधारण मानव
→ human_generic
→ updated

Processing: भक्त
→ human_generic
→ updated

Processing: आवेशावतार
→ mythical_living_being
→ updated

Processing: पत्ता
→ plant
→ updated

Processing: लड़का
→ human_generic
→ updated

Processing: पुरुष
→ human_generic
→ updated

Processing: शुक्ल

In [1]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# CONCEPT TREE
# ============================================================
CONCEPT_LEAF_NODES = [
    "abstract_concept",
    "attribute",
    "state",
    "action_process",
    "measure_quantity",
    "knowledge_linguistic",
    "other"
]

PARENT_MAP = {
    "abstract_concept": ["Concept", "abstract_concept"],
    "attribute": ["Concept", "attribute"],
    "state": ["Concept", "state"],
    "action_process": ["Concept", "action_process"],
    "measure_quantity": ["Concept", "measure_quantity"],
    "knowledge_linguistic": ["Concept", "knowledge_linguistic"],
    "other": ["Concept", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/concept.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class ConceptClassification(BaseModel):
    leaf_category: Literal[
        "abstract_concept",
        "attribute",
        "state",
        "action_process",
        "measure_quantity",
        "knowledge_linguistic",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 8192,
        "temperature": 0.2,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=ConceptClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in CONCEPT_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

concept_entities = file["global_entity_registry"].get("Concept", [])

print(f"Total Concept entities: {len(concept_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in concept_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Concept"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # add only if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Concept entities: 381

Processing: अलौकिक
→ attribute
→ updated

Processing: पाप
→ attribute
→ updated

Processing: माङ्गलिक
→ attribute
→ updated

Processing: अधिकता
→ measure_quantity
→ updated

Processing: कारण
→ abstract_concept
→ updated

Processing: लोक
→ attribute
→ updated

Processing: संग
→ attribute
→ updated

Processing: विद्वान
→ attribute
→ updated

Processing: सत्पुरुष
→ attribute
→ updated

Processing: धनवान
→ attribute
→ updated

Processing: दुरात्मा
→ attribute
→ updated

Processing: कुपात्र
→ attribute
→ updated

Processing: अपात्र
→ attribute
→ updated

Processing: वेदपारगामी
→ attribute
→ updated

Processing: शौकीन
→ attribute
→ updated

Processing: अंत
→ state
→ updated

Processing: काम
→ action_process
→ updated

Processing: आध्यात्मिक विकास
→ abstract_concept
→ updated

Processing: उद्भव
→ action_process
→ updated

Processing: हानि
→ action_process
→ updated

Processing: ज्ञान
→ attribute
→ updated

Processing: अर्चन
→ action_process
→ updated

Processing: 

In [2]:
# %%
import json
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

# ============================================================
# CONFIG
# ============================================================
CLIENT = OpenAI(base_url='http://127.0.0.1:8001/v1', api_key='none')

INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# ============================================================
# GEOGRAPHY TREE
# ============================================================
GEO_LEAF_NODES = [
    "landform",
    "water_body",
    "vegetation_region",
    "atmospheric_region",
    "other"
]

PARENT_MAP = {
    "landform": ["Geographical_Feature", "landform"],
    "water_body": ["Geographical_Feature", "water_body"],
    "vegetation_region": ["Geographical_Feature", "vegetation_region"],
    "atmospheric_region": ["Geographical_Feature", "atmospheric_region"],
    "other": ["Geographical_Feature", "other"]
}

# ============================================================
# SYSTEM PROMPT
# ============================================================
with open("prompts/subtype_categorization/geographical_feature.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

# ============================================================
# SCHEMA
# ============================================================
class GeographyClassification(BaseModel):
    leaf_category: Literal[
        "landform",
        "water_body",
        "vegetation_region",
        "atmospheric_region",
        "other"
    ]

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def call_llm(system_prompt, user_query, response_model=None):
    api_params = {
        "model": "Qwen/Qwen2.5-14B-Instruct",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query},
        ],
        "max_tokens": 512,
        "temperature": 0.1,
    }

    if response_model:
        api_params["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": response_model.__name__,
                "schema": response_model.model_json_schema(),
            },
        }

    response = CLIENT.chat.completions.create(**api_params)
    return response.choices[0].message.content


# ============================================================
# SAFE LLM CALL
# ============================================================
def classify_entity(entity_name):
    user_query = f"Entity: {entity_name}"

    for _ in range(2):
        raw = call_llm(SYSTEM_PROMPT, user_query, response_model=GeographyClassification)

        try:
            parsed = json.loads(raw)
            leaf = parsed["leaf_category"]

            if leaf in GEO_LEAF_NODES:
                return leaf

        except:
            pass

    return "other"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

geo_entities = file["global_entity_registry"].get("Geographical_Feature", [])

print(f"Total Geographical_Feature entities: {len(geo_entities)}")

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

# ============================================================
# PROCESS
# ============================================================
for entity_name in geo_entities:

    print(f"\nProcessing: {entity_name}")

    # ----------------------------
    # get or create node_id
    # ----------------------------
    if entity_name in entity_map:
        node_id = entity_map[entity_name]
        entity = entities[node_id]
    else:
        node_id = f"E_{len(entity_map):06d}"
        entity_map[entity_name] = node_id

        entity = {
            "node_id": node_id,
            "name": entity_name,
            "types": ["Geographical_Feature"]
        }

        entities[node_id] = entity

    # ----------------------------
    # classify
    # ----------------------------
    leaf = classify_entity(entity_name)
    print(f"→ {leaf}")

    full_path = PARENT_MAP[leaf]
    existing_types = set(entity.get("types", []))

    # ----------------------------
    # add only if new
    # ----------------------------
    if set(full_path).issubset(existing_types):
        print("→ no new type to add")
        continue

    existing_types.update(full_path)
    entity["types"] = list(existing_types)

    # ----------------------------
    # save
    # ----------------------------
    atomic_save(entities, ENTITY_FILE)
    atomic_save(entity_map, ENTITY_MAP_FILE)

    print("→ updated")

print("\nDONE.")

Total Geographical_Feature entities: 20

Processing: हिमालय
→ landform
→ updated

Processing: गङ्गा
→ water_body
→ updated

Processing: लोक
→ other
→ updated

Processing: चराचर जगत्
→ other
→ updated

Processing: गंगा तट
→ water_body
→ updated

Processing: समुद्र
→ water_body
→ updated

Processing: आकाश
→ atmospheric_region
→ updated

Processing: जगत्
→ other
→ updated

Processing: गंगा
→ water_body
→ updated

Processing: पर्वत
→ landform
→ updated

Processing: प्रकृति
→ other
→ updated

Processing: नदी
→ water_body
→ updated

Processing: तालाब
→ water_body
→ updated

Processing: सागर
→ water_body
→ updated

Processing: वनस्पति
→ vegetation_region
→ updated

Processing: संसार
→ other
→ updated

Processing: धरती
→ landform
→ updated

Processing: सरोवर
→ water_body
→ updated

Processing: भूमि
→ landform
→ updated

Processing: देवखात
→ other
→ updated

DONE.


In [1]:
# %%
import json
import os
from openai import OpenAI

# ============================================================
# CONFIG
# ============================================================
INPUT_FILE = "processing_state.json"
ENTITY_FILE = "entity_info.json"
ENTITY_MAP_FILE = "entity_id_dict.json"

# Classes already subcategorized (SKIP THESE)
SUBCATEGORIZED_CLASSES = {
    "Time",
    "Activity",
    "Concept",
    "Geographical_Feature",
    "Living_Being",
    "Medical_Concept",
    "Mythical_Entity",
    "Phenomenon"
}

# ============================================================
# UTILS
# ============================================================
def atomic_save(data, path):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)


def load_json(path, default):
    if not os.path.exists(path):
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def generate_node_id(entity_map):
    return f"E_{len(entity_map):06d}"


# ============================================================
# LOAD INPUT
# ============================================================
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    file = json.load(f)

registry = file["global_entity_registry"]

# ============================================================
# LOAD EXISTING OUTPUT
# ============================================================
entities = load_json(ENTITY_FILE, {})
entity_map = load_json(ENTITY_MAP_FILE, {})

print(f"Loaded {len(entity_map)} existing entities")

# ============================================================
# PROCESS
# ============================================================
for base_class, entity_list in registry.items():

    # skip already processed/subcategorized classes
    if base_class in SUBCATEGORIZED_CLASSES:
        continue

    print(f"\nProcessing base class: {base_class} ({len(entity_list)} entities)")

    for entity_name in entity_list:

        # ----------------------------
        # existing entity
        # ----------------------------
        if entity_name in entity_map:
            node_id = entity_map[entity_name]
            entity = entities[node_id]

            existing_types = set(entity.get("types", []))

            if base_class not in existing_types:
                existing_types.add(base_class)
                entity["types"] = list(existing_types)

                print(f"Updated: {entity_name} → +{base_class}")

                atomic_save(entities, ENTITY_FILE)

            continue

        # ----------------------------
        # new entity
        # ----------------------------
        node_id = generate_node_id(entity_map)

        entity_map[entity_name] = node_id

        entities[node_id] = {
            "node_id": node_id,
            "name": entity_name,
            "types": [base_class]
        }

        print(f"Added: {entity_name} → {base_class}")

        atomic_save(entities, ENTITY_FILE)
        atomic_save(entity_map, ENTITY_MAP_FILE)

print("\nDONE.")

Loaded 899 existing entities

Processing base class: Location (42 entities)
Added: द्वारका → Location
Added: त्रिलोकी → Location
Updated: उत्तर → +Location
Added: स्वर्ग → Location
Added: वृन्दावन → Location
Added: मथुरा → Location
Added: नीचे → Location
Added: घर → Location
Added: अवन्तिपुरी → Location
Added: जगह → Location
Added: अर्पण → Location
Updated: लोक → +Location
Added: द्वादशज्योतिर्लिङ्ग → Location
Added: प्रभास → Location
Added: देश → Location
Updated: जगत् → +Location
Added: प्रयाग → Location
Added: भृगुक्षेत्र → Location
Added: परमपद → Location
Updated: नदी → +Location
Added: परलोक → Location
Added: राज्य → Location
Added: पुष्करादितीर्थ → Location
Added: भूलोक → Location
Added: मन्दिर → Location
Added: पद → Location
Added: भारतवर्ष → Location
Added: मण्डप → Location
Added: कूआ → Location
Added: भारत → Location
Added: नरक → Location
Added: काशी → Location
Updated: गंगा → +Location
Added: पूर्वदिशा → Location
Added: स्थान → Location
Added: कोना → Location
Added: कुरुक्षेत

In [21]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import numpy as np

# 1. Setup Model and Tokenizer
# model_name = "BAAI/bge-small-en-v1.5"
model_name = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def get_bge_embeddings(sentences, batch_size=32):
    """
    Generates BGE-M3 embeddings using the transformers library.
    Bypasses multiprocessing bugs by running on the main process.
    """
    all_embeddings = []
    
    # Process in batches for memory efficiency
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        
        # Tokenize
        encoded = tokenizer(
            batch, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**encoded)
            
            # BGE-M3 uses the [CLS] token (index 0) for its dense representation
            # Shape: (batch_size, 1024)
            embeddings = outputs.last_hidden_state[:, 0]
            
            # Unit Normalization (L2) - This is CRITICAL for BGE-M3 
            # to spread out the 'Narrow Cone' of Hindi embeddings.
            embeddings = F.normalize(embeddings, p=2, dim=1)
            
            all_embeddings.append(embeddings.cpu().numpy())
            
    return np.concatenate(all_embeddings, axis=0)

# --- Usage ---
# embeddings = get_bge_embeddings(clean_for_model, batch_size=32)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

embeddings = get_bge_embeddings([
    "relation: incarnate_of    |This avatar is incarnate of this deity", 
    "relation: son_of          |This avatar is son of this deity"
])

score = cosine_similarity(embeddings[0].reshape(1,-1), embeddings[1].reshape(1,-1))
print(score)

[[0.8531288]]


In [23]:
from torch import embedding


embeddings1 = get_bge_embeddings([
    "This avatar is incarnate of this deity", 
    "This deity came to earth as this avatar"
])
print(cosine_similarity(embeddings1[0].reshape(1, -1), embeddings1[1].reshape(1, -1)))
embeddings2 = get_bge_embeddings([
    "relation: incarnate_of ", 
    "relation: came_to_earth "
])
print(cosine_similarity(embeddings2[0].reshape(1, -1), embeddings2[1].reshape(1, -1)))

final_vector = 0.3*embeddings + 0.7*embeddings2
print(cosine_similarity(final_vector[0].reshape(1, -1), final_vector[1].reshape(1, -1)))

[[0.8829771]]
[[0.62920254]]
[[0.74547815]]


In [24]:
from torch import embedding


embeddings1 = get_bge_embeddings([
    "This avatar is incarnate of this deity", 
    "This avatar is son of this deity"
])
print(cosine_similarity(embeddings1[0].reshape(1, -1), embeddings1[1].reshape(1, -1)))
embeddings2 = get_bge_embeddings([
    "relation: incarnate_of ", 
    "relation: son_of "
])
print(cosine_similarity(embeddings2[0].reshape(1, -1), embeddings2[1].reshape(1, -1)))

final_vector = 0.3*embeddings + 0.7*embeddings2
print(cosine_similarity(final_vector[0].reshape(1, -1), final_vector[1].reshape(1, -1)))

[[0.8823644]]
[[0.7679639]]
[[0.7876955]]


month -> comes_after -> month
month -> comes_before -> month

measurement_unit -> bigger_than -> measurement_unit
measurement_unit -> smaller_than -> measurement_unit

measurement_unit -> measures -> concept

nakshatra -> comes_before -> nakshatra
nakshatra -> comes_after -> nakshatra

festival -> falls_on -> all(generic_date, month, moon_phase)
festival -> celebrated_in -> location
festival -> includes_worship_of -> atleast_one(mythical_entity, living_being, geographical_feature, object)

ritual_activity -> is_part_of -> atleast_one(festival, special_date)

disease -> has -> atleast_one(symptom_physical, symptom_mental)
disease -> affects -> Body_part
symptom_physical -> indicates -> disease
symptom_mental -> indicates -> disease
remedy -> alleviates -> atleast_one(disease, symptom_physical, symptom_mental)

celestial_phonomenon -> involves -> Celestial_entity
season -> spans -> month


avatar -> incarnate_of -> deity

(existing 'is_a' and 'is_an' relations will not be modified)

avatar can not have an incoming edge "incarnate_of"
season -> is_a -> month relation not allowed
month -> is_a -> season relation not allowed

Entity
│
├── Time
│   ├── day
│   ├── date
│   │   ├── special_date
│   │   └── generic_date
│   ├── month
│   ├── moon_phase
│   ├── measurement_unit
│   ├── measurement_method
│   ├── nakshatra
│   └── other
│
├── Activity
│   ├── festival
│   ├── ritual_activity
│   ├── mundane_activity
│   └── other
│
├── Location
│
├── Food
│
├── Object
│
├── Text
│
├── Medical_Concept
│   ├── disease
│   ├── symptom_physical
│   ├── symptom_mental
│   ├── secretion_internal
│   ├── secretion_external
│   ├── remedy
│   └── other
│
├── Sanskrit_text
│
├── Social_Group_&_Role
│
├── Phenomenon
│   ├── celestial_phenomenon
│   ├── season
│   ├── natural_phenomenon
│   └── other
│
├── Concept
│   ├── abstract_concept
│   ├── attribute
│   ├── state
│   ├── action_process
│   ├── measure_quantity
│   ├── knowledge_linguistic
│   └── other
│
├── Event
│
├── Primordial_Element
│
├── Geographical_Feature
│   ├── landform
│   ├── water_body
│   ├── vegetation_region
│   ├── atmospheric_region
│   └── other
│
├── Emotions
│
├── Body_part
│
├── Living_Being
│   ├── human_generic
│   ├── human_individual
│   ├── animal
│   ├── plant
│   ├── mythical_living_being
│   └── other
│
├── Celestial_Entity
│
└── Mythical_Entity
    ├── metaphysical_entity
    ├── deity
    ├── avatar
    ├── being_class
    ├── individual_figure
    ├── mythical_creature_object
    └── other